# FPL Point Prediction Model — Training & Evaluation

Trains a LightGBM model (one per position) to predict `total_points` per player per gameweek.

**Pipeline:**
1. Build features from 9 seasons of data (57 features)
2. Run temporal cross-validation (expanding window, 6 folds)
3. Train final model on all data except holdout (2024-25)
4. Save model to disk for use in the RL environment

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path('../data')
MODEL_DIR = Path('../models/point_predictor')

SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25',
]
HOLDOUT = '2024-25'

## 1. Build Features

In [2]:
from fpl_rl.prediction.id_resolver import IDResolver
from fpl_rl.prediction.feature_pipeline import FeaturePipeline

id_resolver = IDResolver(DATA_DIR)
pipeline = FeaturePipeline(DATA_DIR, id_resolver, SEASONS)

print('Building features across all seasons...')
df = pipeline.build()
print(f'\nResult: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Positions: {dict(df["position"].value_counts())}')

Building features across all seasons...


Seasons:   0%|          | 0/9 [00:00<?, ?season/s]

Understat players:   0%|          | 0/683 [00:00<?, ?player/s]

Understat players:   0%|          | 0/647 [00:00<?, ?player/s]

Understat players:   0%|          | 0/624 [00:00<?, ?player/s]

...

Understat players:   0%|          | 0/737 [00:00<?, ?player/s]

Understat players:   0%|          | 0/778 [00:00<?, ?player/s]

Understat players:   0%|          | 0/865 [00:00<?, ?player/s]

Understat players:   0%|          | 0/784 [00:00<?, ?player/s]


Result: 215,087 rows x 63 columns
Seasons: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Positions: {'MID': np.int64(54117), 'DEF': np.int64(42963), 'FWD': np.int64(15661), 'GK': np.int64(13996)}


In [3]:
# Quick data quality check
print('=== Data Quality ===')
print(f'Target (total_points) NaN: {df["target"].isna().sum()}')
print(f'Position NaN: {df["position"].isna().sum()}')
print(f'\nTarget distribution:')
print(df['target'].describe().round(2))

# Feature NaN rates
feature_cols = [c for c in df.columns if c not in {'code','element','season','GW','position','target','total_points'}]
nan_pct = (df[feature_cols].isna().mean() * 100).sort_values(ascending=False)
print(f'\nFeature NaN rates (top 15):')
print(nan_pct.head(15).round(1))

=== Data Quality ===
Target (total_points) NaN: 0
Position NaN: 88350

Target distribution:
count    215087.00
mean          1.31
std           2.53
min          -7.00
25%           0.00
50%           0.00
75%           2.00
max          31.00
Name: target, dtype: float64

Feature NaN rates (top 15):
prev_prog_dist_per90     100.0
opp_pts_conceded_r5      100.0
opp_goals_conceded_r5    100.0
prev_pass_cmp_pct        100.0
prev_blocks_per90        100.0
prev_tkl_int_per90       100.0
prev_gls_per90            97.3
prev_sot_per90            97.3
prev_xg_per90             49.5
prev_shots_per90          49.5
prev_npxg_per90           49.5
prev_xa_per90             49.5
prev_key_passes_per90     49.5
prev_minutes              45.8
xg_rolling_5              34.0
dtype: float64


In [4]:
# Drop rows without position or target
df = df.dropna(subset=['position', 'target'])
print(f'After dropping NaN target/position: {len(df):,} rows')

After dropping NaN target/position: 126,737 rows


## 2. Temporal Cross-Validation

In [5]:
from fpl_rl.prediction.evaluation import TemporalCV

cv = TemporalCV(min_train_seasons=2, holdout=HOLDOUT)
folds = cv.generate_folds(df)
print(f'{len(folds)} folds generated\n')

for i, (train, val, test) in enumerate(folds):
    train_seasons = sorted(train['season'].unique())
    test_season = test['season'].iloc[0]
    print(f'Fold {i+1}: train={train_seasons} ({len(train):,}), '
          f'val={len(val):,}, test={test_season} ({len(test):,})')

2 folds generated

Fold 1: train=['2020-21', '2021-22'] (40,433), val=5,686, test=2022-23 (24,957)
Fold 2: train=['2020-21', '2021-22', '2022-23'] (65,102), val=5,974, test=2023-24 (28,742)


In [6]:
print('Running temporal CV (this may take a minute)...\n')
results = cv.evaluate(df, params={'n_estimators': 500, 'verbose': -1})

print(f'Overall MAE:  {results["mae"]:.3f}')
print(f'Overall RMSE: {results["rmse"]:.3f}')
print(f'\nPer-position MAE:')
for pos, mae in sorted(results['per_position_mae'].items()):
    print(f'  {pos}: {mae:.3f}')

print(f'\nPer-fold results:')
for fold in results['fold_results']:
    print(f'  Fold {fold["fold"]} ({fold["test_season"]}): '
          f'MAE={fold["mae"]:.3f}, RMSE={fold["rmse"]:.3f}, n={fold["n_test"]:,}')

Running temporal CV (this may take a minute)...

Overall MAE:  1.083
Overall RMSE: 2.032

Per-position MAE:
  DEF: 1.164
  FWD: 1.252
  GK: 0.796
  MID: 1.046

Per-fold results:
  Fold 1 (2022-23): MAE=1.159, RMSE=2.094, n=24,957
  Fold 2 (2023-24): MAE=1.018, RMSE=1.977, n=28,742


## 3. Train Final Model

Train on all seasons except the holdout (2024-25), using the last 8 GWs of 2023-24 as validation for early stopping.

In [7]:
from fpl_rl.prediction.model import PointPredictor

# Split train / val
train_seasons = [s for s in SEASONS if s != HOLDOUT]
train_full = df[df['season'].isin(train_seasons)]

last_season = train_seasons[-1]
last_data = train_full[train_full['season'] == last_season]
val_cutoff = last_data['GW'].max() - 8

val_mask = (train_full['season'] == last_season) & (train_full['GW'] > val_cutoff)
val_df = train_full[val_mask]
train_df = train_full[~val_mask]

print(f'Training: {len(train_df):,} rows ({sorted(train_df["season"].unique())})')
print(f'Validation: {len(val_df):,} rows (last 8 GWs of {last_season})')

predictor = PointPredictor(
    params={'n_estimators': 500, 'verbose': -1},
    early_stopping_rounds=50,
)
train_results = predictor.train(train_df, val_df)

print(f'\nTraining MAE per position:')
for pos, mae in sorted(train_results.items()):
    print(f'  {pos}: {mae:.3f}')

Training: 93,018 rows (['2020-21', '2021-22', '2022-23', '2023-24'])
Validation: 6,800 rows (last 8 GWs of 2023-24)

Training MAE per position:
  DEF: 0.968
  FWD: 0.853
  GK: 0.478
  MID: 0.918


In [8]:
# Feature importance (top 20)
fi = predictor.feature_importance()
print('Top 20 features by importance (gain):\n')
print(fi.head(20).to_string(index=False))

Top 20 features by importance (gain):

             feature    importance
      mins_rolling_3 197644.298745
       ict_rolling_3  88709.634465
       selected_norm  32994.636283
        xg_rolling_3  28654.179832
       pts_rolling_3  23872.818166
               value  21200.588843
      ict_rolling_10  19886.897144
       bps_rolling_5  18206.430870
      season_avg_pts  17042.080692
        xa_rolling_3  16027.628390
opp_defence_strength  15775.945038
       ict_rolling_5  15701.391149
   season_total_mins  15151.759416
            was_home  14191.866085
      bps_rolling_10  14088.293963
creativity_rolling_5  13481.952550
          mins_std_5  13179.230008
        prev_minutes  12926.853879
    threat_rolling_5  12888.881599
                 fdr  12442.854454


## 4. Evaluate on Holdout (2024-25)

In [9]:
holdout_df = df[df['season'] == HOLDOUT]

if holdout_df.empty:
    print(f'No holdout data for {HOLDOUT}')
else:
    preds = predictor.predict(holdout_df)
    actuals = holdout_df['target'].values
    errors = np.abs(preds - actuals)

    print(f'Holdout ({HOLDOUT}): {len(holdout_df):,} rows')
    print(f'MAE:  {errors.mean():.3f}')
    print(f'RMSE: {np.sqrt(np.mean((preds - actuals)**2)):.3f}')

    print(f'\nPer-position holdout MAE:')
    for pos in ['GK', 'DEF', 'MID', 'FWD']:
        mask = holdout_df['position'].values == pos
        if mask.any():
            print(f'  {pos}: {errors[mask].mean():.3f} (n={mask.sum():,})')

Holdout (2024-25): 26,919 rows
MAE:  1.020
RMSE: 1.933

Per-position holdout MAE:
  GK: 0.776 (n=2,827)
  DEF: 1.019 (n=9,024)
  MID: 1.031 (n=12,084)
  FWD: 1.209 (n=2,984)


## 5. Save Model

In [10]:
predictor.save(MODEL_DIR)
print(f'Model saved to {MODEL_DIR.resolve()}')
print(f'Files: {[f.name for f in MODEL_DIR.iterdir()]}')

# Verify load roundtrip
loaded = PointPredictor.load(MODEL_DIR)
test_preds = loaded.predict(holdout_df.head(10))
print(f'\nLoad roundtrip OK — sample predictions: {test_preds.round(2)}')

Model saved to C:\Users\alexa\OneDrive\Documents\FPL-RL\models\point_predictor
Files: ['DEF.lgb', 'feature_names.json', 'FWD.lgb', 'GK.lgb', 'metadata.json', 'MID.lgb']

Load roundtrip OK — sample predictions: [0.83 0.13 0.09 0.06 0.06 0.1  0.1  0.06 0.07 0.06]


## 6. Quick Sanity Checks

Spot-check predictions for well-known players.

In [11]:
if not holdout_df.empty:
    holdout_with_preds = holdout_df.copy()
    holdout_with_preds['predicted'] = preds
    holdout_with_preds['error'] = errors

    # Per-GW average prediction vs actual
    gw_summary = holdout_with_preds.groupby('GW').agg(
        avg_actual=('target', 'mean'),
        avg_predicted=('predicted', 'mean'),
        mae=('error', 'mean'),
    ).round(3)
    print('Per-GW summary (holdout):')
    print(gw_summary.to_string())

    # Top predicted players in a sample GW
    sample_gw = holdout_with_preds['GW'].median()
    gw_data = holdout_with_preds[holdout_with_preds['GW'] == sample_gw]
    top = gw_data.nlargest(10, 'predicted')[['code', 'position', 'predicted', 'target']]
    top['name'] = top['code'].apply(id_resolver.player_name)
    print(f'\nTop 10 predicted in GW {int(sample_gw)}:')
    print(top[['name', 'position', 'predicted', 'target']].to_string(index=False))

Per-GW summary (holdout):
    avg_actual  avg_predicted    mae
GW                                  
1        1.317          1.326  1.231
2        1.367          1.378  1.032
3        1.140          1.312  0.888
4        1.235          1.269  1.040
5        1.212          1.274  1.039
6        1.142          1.269  1.019
7        1.311          1.270  1.057
8        1.165          1.240  1.030
9        1.151          1.255  1.018
10       1.150          1.253  0.951
11       1.201          1.233  1.019
12       1.228          1.229  1.140
13       1.200          1.237  1.075
14       1.248          1.219  1.146
15       1.127          1.236  1.023
16       1.168          1.243  1.014
17       1.312          1.229  1.176
18       1.235          1.261  1.024
19       1.176          1.229  1.033
20       1.080          1.244  0.963
21       1.122          1.218  0.941
22       1.167          1.206  1.092
23       1.102          1.227  1.002
24       1.256          1.193  1.117
25       1.2

---

## Usage in the RL Environment

```python
from fpl_rl.env.fpl_env import FPLEnv

env = FPLEnv(
    season='2024-25',
    predictor_model_dir='models/point_predictor',
)
# features[18] in observations now contains predicted points / 15.0
```